# ACMEサーバの切り替え

---

ACME による自動取得 (`use_acme=True`) で構築したCoursewareHubについて、サーバ証明書を取得する ACME サーバを切り替えます。

## 概要

Let's Encrypt の staging で動作確認を行った後に本番へ切り替える場合や、Let's Encrypt と UPKI 電子証明書発行サービスを切り替える場合などに利用します。

ACME サーバを切り替えるには、ACME の設定を変更するだけでなく以下の操作が必要となります。

- 取得済みの証明書を削除する: auth-proxy コンテナは証明書が存在しない場合にのみ新規取得を行います。また取得済みの証明書の更新設定には取得元の ACME サーバが記録されているため、残したままでは切り替え前の ACME サーバから更新され続けます
- CoursewareHub のコンテナを停止して起動しなおす: ACME の設定は Docker の secret として auth-proxy コンテナに渡しています。secret は内容を変更できないため、一度停止して作りなおします
- registry の CA 証明書を更新する: Let's Encrypt の staging に切り替える場合はテスト用 CA の証明書を各ノードの docker と jupyterhub コンテナに信頼させ、それ以外の場合は削除します。jupyterhub コンテナに反映させるため `docker-compose.yml` も生成しなおします

> **注意:** このNotebookの操作では CoursewareHub のコンテナを一度停止するため、操作中は CoursewareHub を利用できません。

> **Let's Encrypt のレート制限:** 本番は同一ドメインに対する証明書の発行回数に制限があります ([Rate Limits](https://letsencrypt.org/docs/rate-limits/))。staging で十分に動作確認を行ってから本番へ切り替えてください。

証明書ファイルの手動配置 (`use_acme=False`) で構築した場合の証明書の更新は「741-サーバ証明書の更新」、取得方式の変更は「743-サーバ証明書の取得方式の変更」で行います。

### グループ名

更新対象となるCoursewareHubを構築したときに指定したansibleのグループ名を指定します。

ansibleのグループ名を確認するために `group_vars`ファイル名の一覧を表示します。

In [ ]:
!ls -1 --hide all group_vars/

上のセルの出力結果を参考にしてグループ名を次のセルに指定してください。

In [ ]:
# (例)
# ugroup_name = 'CoursewareHub'

ugroup_name = 

### チェック

指定されたグループ名が前提条件を満たしていることを確認します。

Ansibleの設定ファイルの場所を環境変数に設定しておきます。

In [ ]:
from pathlib import Path
import os

cfg_ansible = Path('ansible.cfg')
if cfg_ansible.exists():
    os.environ['ANSIBLE_CONFIG'] = str(cfg_ansible.resolve())

構築対象となるVCノードにアクセスできることを確認します。

In [ ]:
target_hub = f'{ugroup_name}_manager'

!ansible {target_hub} -m ping

UnitGroup名に対応する group_varsファイルが存在していることを確認します。

In [ ]:
!test -f group_vars/{ugroup_name}

CoursewareHubのサービスが実行されていることを確認します。

In [ ]:
cmd = "docker stack services --format '{{.Replicas}}' coursewarehub | grep '1/1' | wc -l"
!ansible {target_hub} -m shell -a "{{% raw %}} [ \$({cmd}) -eq 4 ] {{% endraw %}}"

group_varsから現在の設定値を読み込みます。

In [ ]:
import yaml
from pathlib import Path

gvars_path = Path(f'group_vars/{ugroup_name}')
with gvars_path.open() as f:
    gvars = yaml.safe_load(f)

証明書の取得方式が ACME による自動取得 (`use_acme=True`) であることを確認し、現在の ACME サーバを表示します。

In [ ]:
if not gvars.get('use_acme', False):
    raise RuntimeError(
        'ACME による自動取得を利用していません (use_acme=False)。'
        '証明書ファイルの更新は「741-サーバ証明書の更新」、'
        '取得方式の変更は「743-サーバ証明書の取得方式の変更」を利用してください。')

certbot_data_dir = gvars.get('certbot_data_dir', '/jupyter/certbot')
print(f'ACME Server: {gvars.get("acme_server")}')

## 現在の証明書の確認

現在取得している証明書の発行者と有効期限を表示します。

In [ ]:
!ansible {target_hub} -b -a \
    'openssl x509 -noout -subject -issuer -dates -in {certbot_data_dir}/live/{{{{master_fqdn}}}}/fullchain.pem'

## ACME サーバの指定

切り替え先の ACME サーバのパラメータを指定します。各パラメータの詳細は「121-CoursewareHubのセットアップ-ローカルユーザ認証」の「方式A: ACME による自動取得」を参照してください。

### ACME サーバ

切り替え先の ACME サーバの directory URL を `acme_server` に指定します。主な ACME サーバの URL を以下に示します。

| 利用する CA | directory URL | EAB |
| --- | --- | --- |
| Let's Encrypt (テスト / staging) | `https://acme-staging-v02.api.letsencrypt.org/directory` | 不要 |
| Let's Encrypt (本番) | `https://acme-v02.api.letsencrypt.org/directory` | 不要 |
| UPKI 電子証明書発行サービス (NII) | UPKI 運用窓口から提供される URL | 必要 |

In [ ]:
# (例)
# acme_server = 'https://acme-v02.api.letsencrypt.org/directory'           # Let's Encrypt (本番)
# acme_server = 'https://acme-staging-v02.api.letsencrypt.org/directory'   # Let's Encrypt (staging)
# acme_server = 'UPKI 運用窓口から提供された directory URL'                  # UPKI 電子証明書発行サービス

acme_server = ''

### 連絡先メールアドレス

ACME サーバへのアカウント登録に用いるメールアドレスを `acme_email` に指定します。既定値は現在の値です。変更しない場合はそのままで構いません。

In [ ]:
# (例)
# acme_email = 'noc@example.ac.jp'

acme_email = gvars.get('acme_email', '')

### 鍵種別

証明書の鍵種別 (`rsa` または `ecdsa`) を `acme_key_type` に指定します。既定値は現在の値です。変更しない場合はそのままで構いません。

In [ ]:
# (例)
# acme_key_type = 'rsa'
# acme_key_type = 'ecdsa'

acme_key_type = gvars.get('acme_key_type', 'rsa')

### External Account Binding (EAB)

切り替え先の CA が EAB を要求する場合は、次のセルを実行して表示される入力欄に Key ID (`acme_eab_kid`)、HMAC 鍵 (`acme_eab_hmac_key`) の順に入力してください。機密情報のため、入力した値は画面に表示されません。

- UPKI 電子証明書発行サービス: UPKI の管理ポータルで発行された値を入力してください
- Let's Encrypt: EAB は不要です。何も入力せずに Enter キーを押してください

In [ ]:
from getpass import getpass

acme_eab_kid = getpass('ACME EAB KID: ')
acme_eab_hmac_key = getpass('ACME HMAC Key: ')

### 設定値の確認

指定した値を表示して確認します。EAB の Key ID と HMAC 鍵は値を表示せず、入力の有無のみを表示します。

In [ ]:
import sys

if len(acme_server) == 0:
    raise RuntimeError('acme_server を指定してください。')
if acme_server == gvars.get('acme_server'):
    print('(注意) 現在と同じ ACME サーバが指定されています。', file=sys.stderr)
print(f'ACME Server: {gvars.get("acme_server")} -> {acme_server}')
print(f'ACME Email: {acme_email}')
print(f'ACME Key type: {acme_key_type}')
print(f'EAB KID: {"入力あり" if acme_eab_kid else "なし"}')
print(f'EAB HMAC Key: {"入力あり" if acme_eab_hmac_key else "なし"}')

### 事前確認

ACME サーバによる検証の失敗はレート制限の対象として数えられます。コンテナを停止する前に、次のセルに表示されるホスト名と HTTP-01 を [Let's Debug](https://letsdebug.net/) に指定して、問題が検出されないことを確認してください。Let's Debug は、複数の地点からの DNS の名前解決やポート 80 への到達性などを確認するサービスです。

In [ ]:
print(gvars['master_fqdn'])

## 設定の保存

ACME のパラメータを `group_vars` に保存します。

In [ ]:
gvars.update({
    'acme_server': acme_server,
    'acme_email': acme_email,
    'acme_key_type': acme_key_type,
    'acme_eab_kid': acme_eab_kid,
    'acme_eab_hmac_key': acme_eab_hmac_key,
})
with gvars_path.open(mode='w') as f:
    yaml.safe_dump(gvars, stream=f)

## ACME 設定ファイルの更新

保存したパラメータから ACME 設定ファイル (`/opt/coursewarehub/acme-config.sh`) を生成しなおします。まずはチェックモードで確認を行います。

In [ ]:
!ansible-playbook -l {target_hub} -CDv playbooks/setup-certs.yml || true

実際に ACME 設定ファイルの更新を行います。

In [ ]:
!ansible-playbook -l {target_hub} -v playbooks/setup-certs.yml

## コンテナの停止

CoursewareHub のコンテナを停止します。`--detach=false` を指定しているので、全てのサービスが停止するまで待ち合わせを行います。待ち合わせは最大10分で打ち切ります。

In [ ]:
!ansible {target_hub} -a 'timeout 600 docker stack rm --detach=false coursewarehub'

## 取得済みの証明書の削除

切り替え前の ACME サーバから取得した証明書と、その更新設定を削除します。ACME サーバのアカウント情報 (`accounts/`) は ACME サーバごとに分かれているため削除しません。

In [ ]:
!ansible {target_hub} -b -m shell -a \
    'rm -rf {certbot_data_dir}/live/{{{{master_fqdn}}}} \
            {certbot_data_dir}/archive/{{{{master_fqdn}}}} \
            {certbot_data_dir}/renewal/{{{{master_fqdn}}}}.conf'

## registry の CA 証明書の更新

切り替え先の ACME サーバに応じて、各ノードの docker と jupyterhub コンテナに信頼させる CA 証明書を更新します。Let's Encrypt の staging に切り替えた場合はテスト用 CA の証明書を配置し、それ以外に切り替えた場合は配置済みのものを削除します。

jupyterhub コンテナに信頼させるためのファイルはコンテナの起動時に参照されるため、コンテナを起動する前に実行します。また registry からイメージを pull する worker ノードでも必要となるため、managerノードとworkerノードの全てに対して実行します。まずはチェックモードで確認を行います。

In [ ]:
!ansible-playbook -l {ugroup_name} -CDv playbooks/deploy-registry-ca.yml || true

実際に CA 証明書の更新を行います。

In [ ]:
!ansible-playbook -l {ugroup_name} -v playbooks/deploy-registry-ca.yml

## docker-compose.yml の更新

jupyterhub コンテナに CA 証明書をマウントするかどうかは切り替え先の ACME サーバによって変わるため、`docker-compose.yml` を生成しなおします。まずはチェックモードで確認を行います。

In [ ]:
!ansible-playbook -l {target_hub} -CDv playbooks/deploy-docker-compose.yml || true

実際に `docker-compose.yml` の更新を行います。

In [ ]:
!ansible-playbook -l {target_hub} playbooks/deploy-docker-compose.yml

## コンテナの起動

コンテナを起動します。auth-proxy コンテナは起動時に新しい ACME サーバから証明書を取得します。

`--detach=false` を指定しているので、全てのサービスのコンテナが起動するまで待ち合わせを行います。待ち合わせは最大10分で打ち切ります。

> 次のセルの実行結果がエラーとなる場合は、その後のセルを実行してコンテナの実行状況やログを確認してください。

In [ ]:
!ansible {target_hub} -a 'chdir={{{{compose_dir}}}} \
    timeout 600 docker stack deploy --detach=false -c docker-compose.yml coursewarehub'

`auth-proxy` コンテナのログを表示してみます。

> 証明書の取得の失敗などのエラーが表示されていないことを確認してください。

In [ ]:
!ansible {target_hub} -a 'docker service logs --since=30m coursewarehub_auth-proxy'

**証明書の取得に失敗した場合**

auth-proxy コンテナは ACME による証明書の取得に失敗すると終了し、15分後に再起動されて取得をやりなおします。原因を取り除かないまま再試行を続けると ACME サーバのレート制限 (Let's Encrypt の本番では、同じホスト名に対する認証の失敗は1時間に5回まで) に達し、しばらく証明書を取得できなくなります。

上のログに証明書の取得の失敗 (`urn:ietf:params:acme:error` など) が表示されている場合は、次のセルのコメントを外して auth-proxy コンテナを停止してから、以下の方法で原因を調べてください。

- certbot のログ: managerノードの `/jupyter/certbot-logs/letsencrypt.log`
- [Let's Debug](https://letsdebug.net/) に `master_fqdn` と HTTP-01 を指定して、ACME サーバから見た DNS の名前解決とポート 80 への到達性を確認する

原因を取り除いた後に、auth-proxy コンテナを再開してください。

In [ ]:
# 証明書の取得に失敗した場合に auth-proxy コンテナを停止する
# !ansible {target_hub} -a 'docker service scale coursewarehub_auth-proxy=0'

# 原因を取り除いた後に auth-proxy コンテナを再開する
# !ansible {target_hub} -a 'docker service scale coursewarehub_auth-proxy=1'

`registry` コンテナのログを表示してみます。

In [ ]:
!ansible {target_hub} -a 'docker service logs --since=30m coursewarehub_registry'

## 証明書の確認

auth-proxy が 443 番と 5000 番 (registry) で新しい証明書を返していることを確認します。発行者 (`issuer`) が切り替え先の CA になっていることを確認してください。

In [ ]:
for port in [443, 5000]:
    !ansible {target_hub} -m shell -a \
        'openssl s_client -connect {{{{master_fqdn}}}}:{port} -servername {{{{master_fqdn}}}} < /dev/null 2> /dev/null \
        | openssl x509 -noout -subject -issuer -dates'

全てのノードの docker から registry にログインできることを確認します。

In [ ]:
!ansible {ugroup_name} -b -m shell -a \
    'echo {{{{registry_pass}}}} | docker login -u cwh --password-stdin {{{{master_fqdn}}}}:5000'

## CoursewareHubにアクセスする

ACME サーバを切り替えたCoursewareHub環境にアクセスして正しく動作していることを確認してください。ログイン後に、サーバの起動画面と Environments 画面が表示されることも確認してください。これらの画面は jupyterhub コンテナから registry に接続してコースイメージの一覧を取得します。

次のセルを実行するとCoursewareHubのアドレスを表示します。

In [ ]:
print(f'https://{gvars["master_fqdn"]}')